# 02 — Scraper manual de detalle (Bronce)

Visita individualmente cada aviso pendiente (los que ya están en la tabla
`avisos` pero todavía no tienen fila en `avisos_detalle`) y extrae su
descripción completa, características, coordenadas, estado de publicación y
puntos de interés cercanos, tal como los devuelve el sitio, en texto crudo,
sin transformar.

Además, re-chequea periódicamente los avisos que ya están marcados como
`activo` en `avisos`, para detectar si pasaron a pausado, finalizado o si
dejaron de existir. El estado de publicación y su fecha de último chequeo
viven en la tabla `avisos` (no en `avisos_detalle`), para que un aviso que
nunca logra scrapearse igual pueda marcarse como no disponible.

**Alcance de este notebook:**
- Corrida **manual**, sin scheduling.
- Extracción vía regex sobre el texto visible de la página, más un bloque
  JSON embebido para resolver puntos de interés y estado de publicación.
- Todo el scraping ocurre en **pandas puro**; Spark se usa para leer los
  pendientes, actualizar el estado de publicación fila por fila, y para el
  `MERGE INTO` final (upsert) hacia `avisos_detalle`.
- Se detiene de inmediato ante un CAPTCHA, dejando guardado en memoria lo ya
  procesado hasta ese punto, y activa un cooldown (tabla `control`) para que
  la próxima corrida no vuelva a intentar de inmediato.
- Un aviso que falla la extracción de forma persistente (entre corridas, no
  dentro de la misma) queda marcado `no_disponible` tras
  `MAX_INTENTOS_FALLIDOS_DETALLE` fallos, y sale de la cola de pendientes.

**Qué NO hace este notebook:**
- No transforma ningún valor, todo se guarda como texto, tal cual llega.
- No calcula distancias a puntos de referencia (eso ocurre en Oro).
- No resuelve columnas derivadas de fuentes externas (vulnerabilidad).

**Requisito previo:** debe existir al menos un aviso cargado en la tabla
`avisos` antes de correr este notebook (ver `01_scraper_manual_grilla_bronce`,
que también crea el esquema y las tablas si hace falta).

### 1. Configuración y constantes
Delays entre requests, headers, regex de extracción, selectores de
descripción, subcategorías de puntos de interés y radio máximo considerado
"cercano".

In [ ]:
import re
import json
import time
import random
import logging
from datetime import date, datetime, timedelta
from typing import Optional

import requests
import pandas as pd
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

DELAY_MIN = 1.0
DELAY_MAX = 2.0
TIMEOUT_REQUEST_SEG = 5
REINTENTOS_TRAS_ERROR = 2
BACKOFF_REINTENTO_MIN = 3.0
BACKOFF_REINTENTO_MAX = 6.0

# Re-chequeo de avisos activos: cada cuántos días se vuelve a visitar un
# aviso ya scrapeado, y cuántos re-chequeos como máximo por corrida.
DIAS_MIN_ENTRE_RECHEQUEOS = 3
MAX_AVISOS_RECHEQUEO_POR_CORRIDA = 200

# Fallos de scraping CONSECUTIVOS (entre corridas, no dentro de la misma)
# antes de marcar un aviso como 'no_disponible' y sacarlo de la cola de
# pendientes nuevos.
MAX_INTENTOS_FALLIDOS_DETALLE = 5

# Tiempo mínimo de espera antes de reintentar tras un CAPTCHA, mismo valor
# que usa el scraper base del proyecto original.
COOLDOWN_TRAS_CAPTCHA_MINUTOS = 60

BASE_URL = "https://www.portalinmobiliario.com"
OPERACION = "arriendo"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
)

HEADERS_BASE = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
}


def headers_requests(referer: str) -> dict:
    return {**HEADERS_BASE, "Referer": referer}


RE_FECHA_PUBLICACION = re.compile(r"Publicado (hoy|esta semana|hace [^\n\|]+)", re.IGNORECASE)
RE_SUPERFICIE_TOTAL = re.compile(r"Superficie total\s*([\d.,]+)\s*m", re.IGNORECASE)
RE_SUPERFICIE_UTIL = re.compile(r"Superficie útil\s*([\d.,]+)\s*m", re.IGNORECASE)
RE_DORMITORIOS = re.compile(r"Dormitorios\s*(\d+)")
RE_BANOS = re.compile(r"Baños\s*(\d+)")
RE_ESTACIONAMIENTOS = re.compile(r"Estacionamientos:?\s*(\d+)", re.IGNORECASE)
RE_ANTIGUEDAD = re.compile(r"Antigüedad\s*(\d+)\s*años?", re.IGNORECASE)
RE_AMOBLADO = re.compile(r"Amoblado:?\s*(Sí|No)", re.IGNORECASE)
RE_ADMITE_MASCOTAS = re.compile(r"Admite mascotas:?\s*(Sí|No)", re.IGNORECASE)
RE_CONDOMINIO_CERRADO = re.compile(r"En condominio cerrado:?\s*(Sí|No)", re.IGNORECASE)
RE_BODEGAS = re.compile(r"Bodegas\s*(\d+)", re.IGNORECASE)
RE_GASTOS_COMUNES = re.compile(r"Gastos comunes:?\s*\$?\s*([\d.,]+)", re.IGNORECASE)
RE_GASTOS_COMUNES_RESUMEN = re.compile(r"Gastos comunes\s+desde\s*\$?\s*([\d.,]+)", re.IGNORECASE)
RE_ESTACIONAMIENTO_VISITAS = re.compile(r"Estacionamiento de visitas:?\s*(Sí|No)", re.IGNORECASE)
RE_SOLO_FAMILIAS = re.compile(r"Solo familias:?\s*(Sí|No)", re.IGNORECASE)
RE_MAX_HABITANTES = re.compile(r"Cantidad máxima de habitantes\s*(\d+)", re.IGNORECASE)
RE_PISCINA = re.compile(r"Piscina:?\s*(Sí|No)", re.IGNORECASE)
RE_QUINCHO = re.compile(r"Quincho\D{0,15}?:?\s*(Sí|No)", re.IGNORECASE)
RE_CONSERJERIA = re.compile(r"Conserjería:?\s*(Sí|No)", re.IGNORECASE)
RE_ASCENSOR = re.compile(r"Ascensor:?\s*(Sí|No)", re.IGNORECASE)
RE_PISO_UNIDAD = re.compile(r"Número de piso de la unidad\s*(\d+)", re.IGNORECASE)
RE_DEPTOS_POR_PISO = re.compile(r"Departamentos por piso\s*(\d+)", re.IGNORECASE)
RE_LATITUD = re.compile(r'"latitude":"(-?[\d.]+)"')
RE_LONGITUD = re.compile(r'"longitude":"(-?[\d.]+)"')
RE_LATLON_MAPA = re.compile(r"center=(-?[\d.]+)%2C(-?[\d.]+)")

SELECTORES_DESCRIPCION = [
    "[data-testid='core-description'] p",
    "p.ui-pdp-description__content",
    "div.ui-pdp-description",
]

RADIO_MAXIMO_POI_M = 500

SUBCATEGORIAS_POI = {
    "paraderos": "paraderos",
    "estaciones de metro": "estaciones_metro",
    "jardines infantiles": "jardines_infantiles",
    "colegios": "colegios",
    "universidades": "universidades",
    "plazas": "plazas",
    "supermercados": "supermercados",
    "farmacias": "farmacias",
    "centros comerciales": "centros_comerciales",
    "hospitales": "hospitales",
    "clinicas": "clinicas",
}

DIAS_ESTA_SEMANA = 3

CLAVES_ESTADO_PUBLICACION = ("item_status_message", "item_status_short_description_message")

### 2. Crear la tabla `avisos_detalle` (si no existe)
Para que el pipeline se pueda reconstruir desde cero: si el catálogo está
vacío, deja creada `avisos_detalle` con su esquema completo. Si ya existe,
`IF NOT EXISTS` no hace nada. El esquema de `avisos` y de `control` ya se
crea en `01_scraper_manual_grilla_bronce`.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS gran_concepcion.01_bronce.avisos_detalle (
        id_aviso                              STRING NOT NULL,
        descripcion                           STRING,
        fecha_publicacion_texto               STRING,
        fecha_publicacion_aprox               STRING,
        fecha_publicacion_precision           STRING,
        superficie_total_m2                   STRING,
        superficie_util_m2                    STRING,
        dormitorios                           STRING,
        banos                                 STRING,
        estacionamientos                      STRING,
        antiguedad_anos                       STRING,
        amoblado                              STRING,
        admite_mascotas                       STRING,
        condominio_cerrado                    STRING,
        bodegas                               STRING,
        gastos_comunes                        STRING,
        estacionamiento_visitas               STRING,
        solo_familias                         STRING,
        max_habitantes                        STRING,
        piscina                               STRING,
        quincho                               STRING,
        conserjeria                           STRING,
        ascensor                              STRING,
        piso_unidad                           STRING,
        deptos_por_piso                       STRING,
        barrio                                STRING,
        latitud                               STRING,
        longitud                              STRING,
        cantidad_paraderos                    STRING,
        distancia_min_m_paraderos             STRING,
        cantidad_estaciones_metro             STRING,
        distancia_min_m_estaciones_metro      STRING,
        cantidad_jardines_infantiles          STRING,
        distancia_min_m_jardines_infantiles   STRING,
        cantidad_colegios                     STRING,
        distancia_min_m_colegios              STRING,
        cantidad_universidades                STRING,
        distancia_min_m_universidades         STRING,
        cantidad_plazas                       STRING,
        distancia_min_m_plazas                STRING,
        cantidad_supermercados                STRING,
        distancia_min_m_supermercados         STRING,
        cantidad_farmacias                    STRING,
        distancia_min_m_farmacias             STRING,
        cantidad_centros_comerciales          STRING,
        distancia_min_m_centros_comerciales   STRING,
        cantidad_hospitales                   STRING,
        distancia_min_m_hospitales            STRING,
        cantidad_clinicas                     STRING,
        distancia_min_m_clinicas              STRING,
        fecha_scrapeo                         STRING
    )
""")

print("Tabla avisos_detalle verificada/creada.")

### 3. Adaptador HTML
Envuelve el HTML descargado para que se comporte como la interfaz mínima
que necesitan las funciones de extracción (locator + inner_text).

In [0]:
class _LocatorHTMLEstatico:
    def __init__(self, soup, selector):
        self._soup = soup
        self._selector = selector

    @property
    def first(self):
        return self

    def count(self):
        if self._selector == "body":
            return 1
        return len(self._soup.select(self._selector))

    def inner_text(self):
        if self._selector == "body":
            nodo = self._soup.body or self._soup
        else:
            elementos = self._soup.select(self._selector)
            if not elementos:
                return ""
            nodo = elementos[0]
        return nodo.get_text("\n", strip=True)


class PaginaHTMLEstatico:
    def __init__(self, html):
        self._html = html or ""
        self._soup = BeautifulSoup(self._html, "lxml")

    def content(self):
        return self._html

    def locator(self, selector):
        return _LocatorHTMLEstatico(self._soup, selector)

### 4. Funciones de parsing de fecha, descripción, coordenadas y JSON embebido

In [0]:
def parsear_fecha_relativa(texto):
    if not texto:
        return None
    texto_normalizado = texto.strip().lower()
    if texto_normalizado == "hoy":
        return date.today().isoformat()
    if texto_normalizado == "esta semana":
        return (date.today() - timedelta(days=DIAS_ESTA_SEMANA)).isoformat()

    m = re.search(r"(\d+)\s*(día|dias|semana|mes|meses|año|años)", texto, re.IGNORECASE)
    if not m:
        return None
    cantidad = int(m.group(1))
    unidad = m.group(2).lower()

    if "día" in unidad or "dia" in unidad:
        delta = timedelta(days=cantidad)
    elif "semana" in unidad:
        delta = timedelta(weeks=cantidad)
    elif "mes" in unidad:
        delta = timedelta(days=30 * cantidad)
    elif "año" in unidad:
        delta = timedelta(days=365 * cantidad)
    else:
        return None

    return (date.today() - delta).isoformat()


def determinar_precision_fecha(texto):
    if not texto:
        return None
    if texto.strip().lower() == "esta semana":
        return "aproximada_categoria"
    return "exacta"


def extraer_descripcion(page):
    for selector in SELECTORES_DESCRIPCION:
        loc = page.locator(selector)
        if loc.count() > 0:
            try:
                return loc.first.inner_text().strip()
            except Exception:
                continue
    return None


def extraer_coordenadas(page):
    html_completo = page.content()
    m_lat = RE_LATITUD.search(html_completo)
    m_lon = RE_LONGITUD.search(html_completo)
    if m_lat and m_lon:
        return {"latitud": m_lat.group(1), "longitud": m_lon.group(1)}
    m_mapa = RE_LATLON_MAPA.search(html_completo)
    if m_mapa:
        return {"latitud": m_mapa.group(1), "longitud": m_mapa.group(2)}
    return {"latitud": None, "longitud": None}


def normalizar_texto(texto):
    reemplazos = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", "ñ": "n"}
    texto = texto.lower().strip()
    for con_tilde, sin_tilde in reemplazos.items():
        texto = texto.replace(con_tilde, sin_tilde)
    return texto


def parsear_distancia_metros(texto):
    m = re.search(r"([\d.,]+)\s*(metros|km)", texto, re.IGNORECASE)
    if not m:
        return None
    valor = float(m.group(1).replace(".", "").replace(",", "."))
    unidad = m.group(2).lower()
    return valor * 1000 if unidad == "km" else valor


def extraer_json_estado_pagina(page):
    html_completo = page.content()
    m = re.search(r"_n\.ctx\.r=(\{.*?\});_n\.ctx\.r\.assets\.manifest=", html_completo, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None

### 5. Búsqueda recursiva dentro del JSON embebido
Funciones auxiliares para ubicar, dentro del JSON de estado de la página,
las categorías de puntos de interés, valores por clave, y el componente de
estado de publicación.

In [ ]:
def _buscar_categorias_poi(nodo):
    if isinstance(nodo, dict):
        categorias = nodo.get("categories")
        if isinstance(categorias, list) and categorias and isinstance(categorias[0], dict) \
                and "subcategories" in categorias[0]:
            return categorias
        for valor in nodo.values():
            resultado = _buscar_categorias_poi(valor)
            if resultado:
                return resultado
    elif isinstance(nodo, list):
        for item in nodo:
            resultado = _buscar_categorias_poi(item)
            if resultado:
                return resultado
    return None


def _buscar_valor_por_clave(nodo, clave):
    encontrados = []

    def _recorrer(n):
        if isinstance(n, dict):
            if clave in n:
                encontrados.append(n[clave])
            for v in n.values():
                _recorrer(v)
        elif isinstance(n, list):
            for item in n:
                _recorrer(item)

    _recorrer(nodo)
    for valor in encontrados:
        if valor:
            return valor
    return None


def _buscar_nodo_por_clave(nodo, claves):
    """
    Búsqueda recursiva del primer componente (dict) cuyo campo "id" sea
    alguna de `claves`, devolviendo el componente completo (no el valor de
    "id"). En el JSON real de la página cada componente es un dict con forma
    {"id": "item_status_message", "body": {...}, ...}, es decir
    "item_status_message" es el VALOR de la llave "id", no una llave propia
    del dict.
    """
    if isinstance(nodo, dict):
        if nodo.get("id") in claves:
            return nodo
        for valor in nodo.values():
            resultado = _buscar_nodo_por_clave(valor, claves)
            if resultado is not None:
                return resultado
    elif isinstance(nodo, list):
        for item in nodo:
            resultado = _buscar_nodo_por_clave(item, claves)
            if resultado is not None:
                return resultado
    return None


def extraer_puntos_interes(estado):
    resultado = {}
    for clave in SUBCATEGORIAS_POI.values():
        resultado[f"cantidad_{clave}"] = None
        resultado[f"distancia_min_m_{clave}"] = None

    if not estado:
        return resultado

    categorias = _buscar_categorias_poi(estado)
    if not categorias:
        return resultado

    for categoria in categorias:
        for subcategoria in categoria.get("subcategories", []):
            nombre = normalizar_texto(subcategoria.get("title", {}).get("text", ""))
            clave = SUBCATEGORIAS_POI.get(nombre)
            if not clave:
                continue

            items = subcategoria.get("items", [])
            distancias_dentro_del_radio = []
            for item in items:
                texto_subtitulo = item.get("subtitle", {}).get("label", {}).get("text", "")
                distancia = parsear_distancia_metros(texto_subtitulo)
                if distancia is not None and distancia <= RADIO_MAXIMO_POI_M:
                    distancias_dentro_del_radio.append(distancia)

            resultado[f"cantidad_{clave}"] = len(distancias_dentro_del_radio)
            resultado[f"distancia_min_m_{clave}"] = (
                min(distancias_dentro_del_radio) if distancias_dentro_del_radio else None
            )

    return resultado


def extraer_estado_publicacion(estado):
    """
    Busca, dentro de appProps.pageProps.initialState.components.head y
    .short_description, un componente item_status_message /
    item_status_short_description_message. Si no aparece -> 'activo'. Si
    aparece, clasifica su body.text: contiene "pausad" -> 'pausado',
    contiene "finaliz" -> 'finalizado', texto no reconocido -> 'pausado' por
    precaución (la sola presencia del componente ya señala que el aviso no
    está activo con normalidad).
    """
    if not estado or not isinstance(estado, dict):
        return "activo"

    initial_state = estado.get("appProps")
    initial_state = initial_state.get("pageProps") if isinstance(initial_state, dict) else None
    initial_state = initial_state.get("initialState") if isinstance(initial_state, dict) else None

    components = initial_state.get("components") \
        if isinstance(initial_state, dict) and isinstance(initial_state.get("components"), dict) \
        else {}
    subarboles = [components.get("head"), components.get("short_description")]

    for subarbol in subarboles:
        if not subarbol:
            continue
        componente = _buscar_nodo_por_clave(subarbol, CLAVES_ESTADO_PUBLICACION)
        if not componente:
            continue

        texto = ""
        if isinstance(componente, dict):
            body = componente.get("body")
            if isinstance(body, dict):
                texto = body.get("text") or ""

        texto_normalizado = texto.lower()
        if "pausad" in texto_normalizado:
            return "pausado"
        if "finaliz" in texto_normalizado:
            return "finalizado"

        log.warning(f"Estado de publicación con texto no reconocido: '{texto}'. Se marca 'pausado'.")
        return "pausado"

    return "activo"


def _url_coincide_con_aviso(html, id_aviso):
    """
    Confirma que la página descargada realmente corresponde a `id_aviso`, y
    no a otra página (ej. el buscador) a la que el sitio a veces redirige
    cuando el aviso ya fue eliminado, sin devolver un status HTTP distinto de
    200. Exige que TANTO <link rel="canonical"> COMO <meta property="og:url">
    contengan `id_aviso`: son dos tags independientes del <head>, así que
    pedir que ambos coincidan reduce el riesgo de un falso positivo.
    """
    soup = BeautifulSoup(html or "", "lxml")

    canonical = soup.find("link", attrs={"rel": "canonical"})
    canonical_href = canonical.get("href", "") if canonical else ""

    og_url_tag = soup.find("meta", attrs={"property": "og:url"})
    og_url = og_url_tag.get("content", "") if og_url_tag else ""

    return id_aviso in canonical_href and id_aviso in og_url

### 6. Extracción completa de un aviso, y detección de CAPTCHA

In [0]:
def extraer_detalle(page):
    texto_completo = page.locator("body").inner_text()

    def buscar(patron):
        m = patron.search(texto_completo)
        return m.group(1).strip() if m else None

    gastos_comunes = buscar(RE_GASTOS_COMUNES)
    if gastos_comunes is None:
        gastos_comunes = buscar(RE_GASTOS_COMUNES_RESUMEN)

    fecha_texto = buscar(RE_FECHA_PUBLICACION)
    if fecha_texto and fecha_texto.lower().startswith("hace "):
        fecha_texto = fecha_texto[len("hace "):]

    coordenadas = extraer_coordenadas(page)
    estado = extraer_json_estado_pagina(page)
    puntos_interes = extraer_puntos_interes(estado)
    barrio = _buscar_valor_por_clave(estado, "neighborhood") if estado else None
    estado_publicacion = extraer_estado_publicacion(estado)

    return {
        "descripcion": extraer_descripcion(page),
        "fecha_publicacion_texto": fecha_texto,
        "fecha_publicacion_aprox": parsear_fecha_relativa(fecha_texto),
        "fecha_publicacion_precision": determinar_precision_fecha(fecha_texto),
        "superficie_total_m2": buscar(RE_SUPERFICIE_TOTAL),
        "superficie_util_m2": buscar(RE_SUPERFICIE_UTIL),
        "dormitorios": buscar(RE_DORMITORIOS),
        "banos": buscar(RE_BANOS),
        "estacionamientos": buscar(RE_ESTACIONAMIENTOS),
        "antiguedad_anos": buscar(RE_ANTIGUEDAD),
        "amoblado": buscar(RE_AMOBLADO),
        "admite_mascotas": buscar(RE_ADMITE_MASCOTAS),
        "condominio_cerrado": buscar(RE_CONDOMINIO_CERRADO),
        "bodegas": buscar(RE_BODEGAS),
        "gastos_comunes": gastos_comunes,
        "estacionamiento_visitas": buscar(RE_ESTACIONAMIENTO_VISITAS),
        "solo_familias": buscar(RE_SOLO_FAMILIAS),
        "max_habitantes": buscar(RE_MAX_HABITANTES),
        "piscina": buscar(RE_PISCINA),
        "quincho": buscar(RE_QUINCHO),
        "conserjeria": buscar(RE_CONSERJERIA),
        "ascensor": buscar(RE_ASCENSOR),
        "piso_unidad": buscar(RE_PISO_UNIDAD),
        "deptos_por_piso": buscar(RE_DEPTOS_POR_PISO),
        "latitud": coordenadas["latitud"],
        "longitud": coordenadas["longitud"],
        "barrio": barrio,
        "estado_publicacion": estado_publicacion,
        **puntos_interes,
    }


def hay_captcha(page):
    contenido = page.content().lower()
    if "captcha" not in contenido[:8000]:
        return False
    texto = page.locator("body").inner_text()
    parece_contenido_real = bool(
        RE_SUPERFICIE_TOTAL.search(texto)
        or RE_SUPERFICIE_UTIL.search(texto)
        or RE_DORMITORIOS.search(texto)
    )
    return not parece_contenido_real


def construir_referer(comuna, tipo_propiedad):
    return f"{BASE_URL}/{OPERACION}/{tipo_propiedad}/{comuna}"

### 7. Descarga con reintento
Visita la URL de un aviso, con reintentos automáticos ante fallos
transitorios, y devuelve el resultado de la extracción completa.

In [ ]:
def obtener_detalle_aviso(url, id_aviso, comuna, tipo_propiedad):
    """
    Devuelve "resultado": "ok" | "captcha" | "no_encontrado" | "error".
    "no_encontrado" es distinto de "error": el fetch funcionó (status 200,
    sin CAPTCHA) pero la página descargada no corresponde a `id_aviso` (ver
    `_url_coincide_con_aviso`), típicamente porque el aviso ya no existe y el
    sitio redirigió a otra página sin devolver un status distinto de 200. No
    tiene sentido reintentar la misma URL, así que se resuelve en el primer
    intento, sin pasar por el loop de reintentos.
    """
    referer = construir_referer(comuna, tipo_propiedad)
    intentos_totales = 1 + REINTENTOS_TRAS_ERROR
    ultimo_status = None
    ultimo_motivo = None

    for intento in range(1, intentos_totales + 1):
        try:
            resp = requests.get(url, headers=headers_requests(referer), timeout=TIMEOUT_REQUEST_SEG)
            ultimo_status = resp.status_code

            if resp.status_code != 200:
                ultimo_motivo = f"status HTTP {resp.status_code}"
                raise ValueError(ultimo_motivo)

            pagina = PaginaHTMLEstatico(resp.text)

            if hay_captcha(pagina):
                return {"resultado": "captcha", "status_http": ultimo_status, "motivo": "captcha"}

            if not _url_coincide_con_aviso(resp.text, id_aviso):
                log.warning(f"La página descargada para {id_aviso} ({url}) no corresponde a ese aviso "
                            f"(canonical/og:url no lo mencionan), probablemente fue eliminado.")
                return {
                    "resultado": "no_encontrado", "status_http": ultimo_status,
                    "motivo": "la página devuelta no corresponde al aviso (probablemente eliminado)",
                }

            datos = extraer_detalle(pagina)
            return {"resultado": "ok", "datos": datos, "status_http": ultimo_status, "motivo": None}

        except (requests.RequestException, ValueError) as e:
            ultimo_motivo = ultimo_motivo or str(e)
            if intento < intentos_totales:
                espera = random.uniform(BACKOFF_REINTENTO_MIN, BACKOFF_REINTENTO_MAX)
                log.warning(f"Intento {intento}/{intentos_totales} falló para {url} ({ultimo_motivo}). "
                            f"Reintentando en {espera:.1f}s...")
                time.sleep(espera)
            else:
                log.warning(f"Agotados los {intentos_totales} intentos para {url} ({ultimo_motivo}).")

    return {"resultado": "error", "status_http": ultimo_status, "motivo": ultimo_motivo}

### 8. Funciones de persistencia incremental
Cooldown tras CAPTCHA (tabla `control`), contador de fallos persistentes y
estado de publicación (ambos en `avisos`). Se aplican fila por fila, a
medida que se visita cada aviso, en vez de esperar al final de la corrida:
así un aviso que falla queda registrado aunque la corrida se corte después
por CAPTCHA.

In [ ]:
def _sql_str(valor):
    """Escapa comillas simples para armar literales SQL seguros a partir de
    id_aviso (siempre con forma MLC-<dígitos>, pero se escapa igual por si acaso)."""
    return str(valor).replace("'", "''")


def leer_control(clave):
    fila = spark.sql(f"""
        SELECT valor FROM gran_concepcion.01_bronce.control WHERE clave = '{_sql_str(clave)}'
    """).collect()
    return fila[0]["valor"] if fila else None


def escribir_control(clave, valor):
    spark.sql(f"""
        MERGE INTO gran_concepcion.01_bronce.control AS c
        USING (SELECT '{_sql_str(clave)}' AS clave, '{_sql_str(valor)}' AS valor) AS nuevo
        ON c.clave = nuevo.clave
        WHEN MATCHED THEN UPDATE SET c.valor = nuevo.valor
        WHEN NOT MATCHED THEN INSERT (clave, valor) VALUES (nuevo.clave, nuevo.valor)
    """)


def tiempo_restante_cooldown():
    valor = leer_control("ultimo_captcha_detalle")
    if not valor:
        return None
    ultimo_captcha = datetime.fromisoformat(valor)
    transcurrido = datetime.now() - ultimo_captcha
    cooldown_total = timedelta(minutes=COOLDOWN_TRAS_CAPTCHA_MINUTOS)
    return (cooldown_total - transcurrido) if transcurrido < cooldown_total else None


def registrar_captcha():
    escribir_control("ultimo_captcha_detalle", datetime.now().isoformat())


def actualizar_estado_publicacion(id_aviso, estado_publicacion):
    spark.sql(f"""
        UPDATE gran_concepcion.01_bronce.avisos
        SET estado_publicacion = '{_sql_str(estado_publicacion)}',
            fecha_ultimo_chequeo_estado = current_date()
        WHERE id_aviso = '{_sql_str(id_aviso)}'
    """)


def incrementar_intentos_fallidos_detalle(id_aviso):
    spark.sql(f"""
        UPDATE gran_concepcion.01_bronce.avisos
        SET intentos_fallidos_detalle = COALESCE(intentos_fallidos_detalle, 0) + 1
        WHERE id_aviso = '{_sql_str(id_aviso)}'
    """)
    fila = spark.sql(f"""
        SELECT intentos_fallidos_detalle FROM gran_concepcion.01_bronce.avisos
        WHERE id_aviso = '{_sql_str(id_aviso)}'
    """).collect()
    return fila[0]["intentos_fallidos_detalle"] if fila else 0


def resetear_intentos_fallidos_detalle(id_aviso):
    spark.sql(f"""
        UPDATE gran_concepcion.01_bronce.avisos
        SET intentos_fallidos_detalle = 0
        WHERE id_aviso = '{_sql_str(id_aviso)}'
    """)


def visitar_aviso(fila, es_rechequeo, resultados_acumulados):
    """
    Visita un aviso (nuevo o re-chequeo), y aplica las consecuencias de
    persistencia según el resultado. Devuelve 'ok', 'cambio_estado' (solo
    relevante en re-chequeo), 'no_disponible', 'captcha' o 'error'. Si el
    resultado es exitoso, agrega la fila (sin `estado_publicacion`, que se
    guarda aparte en `avisos`) a `resultados_acumulados` para el MERGE final.
    """
    id_aviso = fila["id_aviso"]
    url = fila["url"]
    log.info(f"{'Re-chequeando' if es_rechequeo else 'Visitando'} {id_aviso}: {url}")

    resultado = obtener_detalle_aviso(url, id_aviso, fila["comuna"], fila["tipo_propiedad"])

    if resultado["resultado"] == "captcha":
        log.error(f"CAPTCHA detectado en {url}. Deteniendo la corrida ahora mismo. "
                  f"Cooldown de {COOLDOWN_TRAS_CAPTCHA_MINUTOS} minutos antes de la próxima.")
        registrar_captcha()
        return "captcha"

    if resultado["resultado"] == "no_encontrado":
        actualizar_estado_publicacion(id_aviso, "no_disponible")
        log.warning(f"{id_aviso}: la página descargada no corresponde a este aviso ({resultado['motivo']}). "
                    f"Se marca estado_publicacion='no_disponible' de inmediato.")
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        return "no_disponible"

    if resultado["resultado"] == "error":
        nuevo_contador = incrementar_intentos_fallidos_detalle(id_aviso)
        log.warning(f"No se pudo obtener {id_aviso} tras reintentos ({resultado['motivo']}). "
                    f"intentos_fallidos_detalle={nuevo_contador}.")
        if nuevo_contador > MAX_INTENTOS_FALLIDOS_DETALLE:
            actualizar_estado_publicacion(id_aviso, "no_disponible")
            log.warning(f"{id_aviso} superó MAX_INTENTOS_FALLIDOS_DETALLE={MAX_INTENTOS_FALLIDOS_DETALLE} "
                        f"fallos consecutivos. Se marca estado_publicacion='no_disponible'.")
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        return "error"

    datos = dict(resultado["datos"])
    estado_publicacion = datos.pop("estado_publicacion", "activo")
    datos["id_aviso"] = id_aviso
    datos["fecha_scrapeo"] = date.today().isoformat()
    resultados_acumulados.append(datos)

    actualizar_estado_publicacion(id_aviso, estado_publicacion)
    resetear_intentos_fallidos_detalle(id_aviso)

    log.info(f"  -> Guardado. estado_publicacion={estado_publicacion}")
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    if es_rechequeo and estado_publicacion != "activo":
        return "cambio_estado"
    return "ok"

### 9. Chequear cooldown tras CAPTCHA
Si un CAPTCHA reciente dejó un cooldown activo (tabla `control`), el
notebook se detiene acá mismo sin visitar ningún aviso, para no reintentar
de inmediato y arriesgar otro bloqueo.

In [ ]:
restante = tiempo_restante_cooldown()
if restante:
    minutos = int(restante.total_seconds() // 60) + 1
    log.warning(f"En cooldown tras un CAPTCHA reciente. Faltan ~{minutos} minutos. "
                f"No se hace nada en esta corrida.")
    dbutils.notebook.exit(f"Cooldown activo, faltan ~{minutos} minutos.")
else:
    log.info("Sin cooldown activo, se continúa con la corrida.")

### 10. Identificar avisos pendientes de detalle (nuevos)
LEFT JOIN entre `avisos` y `avisos_detalle` en Bronce: trae solo los avisos
que todavía no tienen su detalle scrapeado y que no fueron marcados
`no_disponible` en una corrida anterior (esos ya agotaron sus reintentos y
no se vuelven a pedir).

In [ ]:
pendientes_nuevos_rows = spark.sql("""
    SELECT a.id_aviso, a.url, a.comuna, a.tipo_propiedad
    FROM gran_concepcion.01_bronce.avisos a
    LEFT JOIN gran_concepcion.01_bronce.avisos_detalle d ON a.id_aviso = d.id_aviso
    WHERE d.id_aviso IS NULL
      AND a.url IS NOT NULL
      AND (a.estado_publicacion IS NULL OR a.estado_publicacion != 'no_disponible')
""").collect()

pendientes_nuevos = [row.asDict() for row in pendientes_nuevos_rows]
log.info(f"{len(pendientes_nuevos)} avisos nuevos pendientes de detalle.")

### 11. Scraping de detalle, en memoria (nuevos)
Visita cada aviso nuevo pendiente, uno por uno, con delay aleatorio entre
requests. Se detiene de inmediato si detecta CAPTCHA. Las consecuencias de
cada resultado (estado de publicación, contador de fallos) se aplican
dentro de `visitar_aviso`, fila por fila.

In [ ]:
resultados_nuevos = []
procesados_nuevos = 0
detenido_por_captcha = False

for fila in pendientes_nuevos:
    resultado = visitar_aviso(fila, es_rechequeo=False, resultados_acumulados=resultados_nuevos)

    if resultado == "captcha":
        detenido_por_captcha = True
        break
    if resultado == "ok":
        procesados_nuevos += 1

    log.info(f"  -> ({procesados_nuevos + 1}/{len(pendientes_nuevos)} intentados)")

log.info(f"Nuevos: procesados {procesados_nuevos} de {len(pendientes_nuevos)}. "
          f"Detenido por CAPTCHA: {detenido_por_captcha}")

### 12. Identificar avisos pendientes de re-chequeo
Avisos `activo` nunca chequeados (`fecha_ultimo_chequeo_estado IS NULL`) o
que superaron `DIAS_MIN_ENTRE_RECHEQUEOS` desde su último chequeo. Los NULL
quedan primero (son los más urgentes), y entre ellos el orden ascendente por
fecha ya deja los vencidos más antiguos primero, sin necesitar un CASE
aparte. Solo corre si no se cortó por CAPTCHA en la sección anterior.

In [ ]:
if detenido_por_captcha:
    pendientes_rechequeo = []
    log.info("Se saltea el re-chequeo esta corrida: ya se detuvo por CAPTCHA en la sección anterior.")
else:
    pendientes_rechequeo_rows = spark.sql(f"""
        SELECT id_aviso, url, comuna, tipo_propiedad, fecha_ultimo_chequeo_estado
        FROM gran_concepcion.01_bronce.avisos
        WHERE estado_publicacion = 'activo'
          AND url IS NOT NULL
          AND (
              fecha_ultimo_chequeo_estado IS NULL
              OR fecha_ultimo_chequeo_estado <= date_sub(current_date(), {DIAS_MIN_ENTRE_RECHEQUEOS})
          )
        ORDER BY fecha_ultimo_chequeo_estado ASC NULLS FIRST
        LIMIT {MAX_AVISOS_RECHEQUEO_POR_CORRIDA}
    """).collect()
    pendientes_rechequeo = [row.asDict() for row in pendientes_rechequeo_rows]

log.info(f"{len(pendientes_rechequeo)} avisos activos pendientes de re-chequeo.")

### 13. Scraping de re-chequeo, en memoria
Mismo mecanismo que la sección anterior, marcando `es_rechequeo=True` para
que `visitar_aviso` cuente los cambios de estado detectados.

In [ ]:
resultados_rechequeo = []
rechequeos_procesados = 0
cambios_estado = 0

for fila in pendientes_rechequeo:
    resultado = visitar_aviso(fila, es_rechequeo=True, resultados_acumulados=resultados_rechequeo)

    if resultado == "captcha":
        detenido_por_captcha = True
        break
    if resultado in ("ok", "cambio_estado"):
        rechequeos_procesados += 1
    if resultado in ("cambio_estado", "no_disponible"):
        cambios_estado += 1

log.info(f"Re-chequeo: procesados {rechequeos_procesados} de {len(pendientes_rechequeo)}. "
          f"Cambios de estado detectados: {cambios_estado}. Detenido por CAPTCHA: {detenido_por_captcha}")

### 14. Armar DataFrame combinado (nuevos + re-chequeo)
Sin ninguna conversión de tipo, todo queda como texto, tal cual llegó.
Combina ambos resultados porque el MERGE final debe insertar los avisos
nuevos y actualizar los re-chequeados en una sola pasada.

In [ ]:
df_detalle = pd.DataFrame(resultados_nuevos + resultados_rechequeo)
print(f"{len(df_detalle)} filas para upsert ({len(resultados_nuevos)} nuevas, {len(resultados_rechequeo)} re-chequeadas)")
df_detalle.head()

### 15. Crear vista temporal para el upsert

In [ ]:
spark.createDataFrame(df_detalle).createOrReplaceTempView("detalle_upsert_tmp")

### 16. MERGE final hacia Bronce (upsert)
Inserta los avisos nuevos y actualiza los re-chequeados. Nunca toca
`estado_publicacion` (vive en `avisos`) ni las columnas de vulnerabilidad,
que no existen en esta tabla.

In [ ]:
%sql
MERGE INTO gran_concepcion.01_bronce.avisos_detalle AS detalle
USING detalle_upsert_tmp AS nuevos
ON detalle.id_aviso = nuevos.id_aviso
WHEN MATCHED THEN UPDATE SET
    descripcion = nuevos.descripcion,
    fecha_publicacion_texto = nuevos.fecha_publicacion_texto,
    fecha_publicacion_aprox = nuevos.fecha_publicacion_aprox,
    fecha_publicacion_precision = nuevos.fecha_publicacion_precision,
    superficie_total_m2 = nuevos.superficie_total_m2,
    superficie_util_m2 = nuevos.superficie_util_m2,
    dormitorios = nuevos.dormitorios,
    banos = nuevos.banos,
    estacionamientos = nuevos.estacionamientos,
    antiguedad_anos = nuevos.antiguedad_anos,
    amoblado = nuevos.amoblado,
    admite_mascotas = nuevos.admite_mascotas,
    condominio_cerrado = nuevos.condominio_cerrado,
    bodegas = nuevos.bodegas,
    gastos_comunes = nuevos.gastos_comunes,
    estacionamiento_visitas = nuevos.estacionamiento_visitas,
    solo_familias = nuevos.solo_familias,
    max_habitantes = nuevos.max_habitantes,
    piscina = nuevos.piscina,
    quincho = nuevos.quincho,
    conserjeria = nuevos.conserjeria,
    ascensor = nuevos.ascensor,
    piso_unidad = nuevos.piso_unidad,
    deptos_por_piso = nuevos.deptos_por_piso,
    barrio = nuevos.barrio,
    latitud = nuevos.latitud,
    longitud = nuevos.longitud,
    cantidad_paraderos = nuevos.cantidad_paraderos,
    distancia_min_m_paraderos = nuevos.distancia_min_m_paraderos,
    cantidad_estaciones_metro = nuevos.cantidad_estaciones_metro,
    distancia_min_m_estaciones_metro = nuevos.distancia_min_m_estaciones_metro,
    cantidad_jardines_infantiles = nuevos.cantidad_jardines_infantiles,
    distancia_min_m_jardines_infantiles = nuevos.distancia_min_m_jardines_infantiles,
    cantidad_colegios = nuevos.cantidad_colegios,
    distancia_min_m_colegios = nuevos.distancia_min_m_colegios,
    cantidad_universidades = nuevos.cantidad_universidades,
    distancia_min_m_universidades = nuevos.distancia_min_m_universidades,
    cantidad_plazas = nuevos.cantidad_plazas,
    distancia_min_m_plazas = nuevos.distancia_min_m_plazas,
    cantidad_supermercados = nuevos.cantidad_supermercados,
    distancia_min_m_supermercados = nuevos.distancia_min_m_supermercados,
    cantidad_farmacias = nuevos.cantidad_farmacias,
    distancia_min_m_farmacias = nuevos.distancia_min_m_farmacias,
    cantidad_centros_comerciales = nuevos.cantidad_centros_comerciales,
    distancia_min_m_centros_comerciales = nuevos.distancia_min_m_centros_comerciales,
    cantidad_hospitales = nuevos.cantidad_hospitales,
    distancia_min_m_hospitales = nuevos.distancia_min_m_hospitales,
    cantidad_clinicas = nuevos.cantidad_clinicas,
    distancia_min_m_clinicas = nuevos.distancia_min_m_clinicas,
    fecha_scrapeo = nuevos.fecha_scrapeo
WHEN NOT MATCHED THEN INSERT (
    id_aviso, descripcion, fecha_publicacion_texto, fecha_publicacion_aprox,
    fecha_publicacion_precision, superficie_total_m2, superficie_util_m2,
    dormitorios, banos, estacionamientos, antiguedad_anos,
    amoblado, admite_mascotas, condominio_cerrado, bodegas, gastos_comunes,
    estacionamiento_visitas, solo_familias, max_habitantes, piscina, quincho,
    conserjeria, ascensor, piso_unidad, deptos_por_piso, barrio,
    latitud, longitud,
    cantidad_paraderos, distancia_min_m_paraderos,
    cantidad_estaciones_metro, distancia_min_m_estaciones_metro,
    cantidad_jardines_infantiles, distancia_min_m_jardines_infantiles,
    cantidad_colegios, distancia_min_m_colegios,
    cantidad_universidades, distancia_min_m_universidades,
    cantidad_plazas, distancia_min_m_plazas,
    cantidad_supermercados, distancia_min_m_supermercados,
    cantidad_farmacias, distancia_min_m_farmacias,
    cantidad_centros_comerciales, distancia_min_m_centros_comerciales,
    cantidad_hospitales, distancia_min_m_hospitales,
    cantidad_clinicas, distancia_min_m_clinicas,
    fecha_scrapeo
) VALUES (
    nuevos.id_aviso, nuevos.descripcion, nuevos.fecha_publicacion_texto,
    nuevos.fecha_publicacion_aprox, nuevos.fecha_publicacion_precision,
    nuevos.superficie_total_m2, nuevos.superficie_util_m2,
    nuevos.dormitorios, nuevos.banos, nuevos.estacionamientos, nuevos.antiguedad_anos,
    nuevos.amoblado, nuevos.admite_mascotas, nuevos.condominio_cerrado,
    nuevos.bodegas, nuevos.gastos_comunes, nuevos.estacionamiento_visitas,
    nuevos.solo_familias, nuevos.max_habitantes, nuevos.piscina, nuevos.quincho,
    nuevos.conserjeria, nuevos.ascensor, nuevos.piso_unidad, nuevos.deptos_por_piso,
    nuevos.barrio, nuevos.latitud, nuevos.longitud,
    nuevos.cantidad_paraderos, nuevos.distancia_min_m_paraderos,
    nuevos.cantidad_estaciones_metro, nuevos.distancia_min_m_estaciones_metro,
    nuevos.cantidad_jardines_infantiles, nuevos.distancia_min_m_jardines_infantiles,
    nuevos.cantidad_colegios, nuevos.distancia_min_m_colegios,
    nuevos.cantidad_universidades, nuevos.distancia_min_m_universidades,
    nuevos.cantidad_plazas, nuevos.distancia_min_m_plazas,
    nuevos.cantidad_supermercados, nuevos.distancia_min_m_supermercados,
    nuevos.cantidad_farmacias, nuevos.distancia_min_m_farmacias,
    nuevos.cantidad_centros_comerciales, nuevos.distancia_min_m_centros_comerciales,
    nuevos.cantidad_hospitales, nuevos.distancia_min_m_hospitales,
    nuevos.cantidad_clinicas, nuevos.distancia_min_m_clinicas,
    nuevos.fecha_scrapeo
)